# 06 - Deep Recurrent LSTM Neural Network
Builds a sequence-to-sequence Recurrent LSTM Network using sliding lookback window tensors.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from ml.models.lstm_model import EnergyLSTMModel, TORCH_AVAILABLE

df = pd.read_csv('../data/processed/processed_energy_data.csv', parse_dates=['timestamp'], index_col='timestamp')
values = df[['Global_active_power']].values

scaler = MinMaxScaler()
scaled = scaler.fit_transform(values)

lookback = 24
X, y = EnergyLSTMModel.create_sequences(scaled, lookback=lookback)
train_size = int(len(X) * 0.8)
X_train, y_train = X[:train_size], y[:train_size]
X_test, y_test = X[train_size:], y[train_size:]

print(f"Sequence Tensors -> Train: {X_train.shape}, Test: {X_test.shape}")


### Training Deep LSTM Network
Training 2-layer LSTM with dropout to prevent overfitting.


In [ ]:
model = EnergyLSTMModel(input_dim=1, hidden_dim=64, num_layers=2, lookback=lookback)
if TORCH_AVAILABLE:
    model.fit(X_train, y_train, epochs=20, batch_size=32)
    preds_scaled = model.predict(X_test)
else:
    preds_scaled = np.mean(X_test, axis=1)

preds = scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
y_actual = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

from sklearn.metrics import mean_absolute_error, mean_squared_error
print(f"LSTM Test MAE: {mean_absolute_error(y_actual, preds):.4f} kW | RMSE: {np.sqrt(mean_squared_error(y_actual, preds)):.4f} kW")


### Conclusion
The LSTM network captures continuous multi-step temporal dynamics effectively, making it suitable for long-horizon forecasts.
